# 리포트 18 — 가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다

> ### 한 일
> **상용 고주파 솔버의 순서 그대로 광선으로 조명면을 찾고 그 면 위에서 부품별 재질 PO 를 적분해 σ 를 냈다.**

### 결과
1. 첫 충돌 탐색과 가림은 Sionna 가 이미 들고 있는 Mitsuba/OptiX 엔진이 하고, 표면전류 적분과 σ 출력은 우리가 얹는다 — 그 문서에 `physical optics` 는 0 회 [^1] 나온다.
2. 조명원을 방위 280° [^2] · 고각 15° [^3] 에 두면 조명원을 향한 외피의 29 [^4]~47% [^5] 가 기체 자신에 가려 있다.
3. «가림 [dB]» 은 **두 팔의 차**다 — P(순수 PO · 점구름 λ/7 · 가림 없음)의 방위평균 σ 에서 B(SBR+PO · 광선격자 λ/12)를 **불투명**으로 돌린 값을 뺀 것이 최대 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)이고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 차가 이산화 바닥(P 팔 한쪽을 λ/7↔λ/12 로 돌린 폭, 최대 0.071 dB [^11]) 위에 있다 — 원장이 «가림 효과» 라고 부르기 위해 건 필요조건을 일곱 기체가 전부 통과한다 [^12].
4. 금속 4그룹만 남긴 메쉬의 방위평균 σ 가 전체의 112% [^13] 다 — 코히런트 합이라 100 % 를 넘는다.
5. 그 광선 격자를 자세마다 다시 정의하면 로터 사이에 가짜 결합이 생긴다 — 가산성 잔차가 격자를 얼렸을 때 8.3e-16 [^14] (기계정밀도)이고 움직이는 격자에서 1.78 [^15] 다. 얼리면 **대역밖**(블레이드 끝 도플러 1229 Hz [^16] 보다 높은 주파수 전부) 절대 전력이 λ/12 에서 13.1 dB [^17] 내려간다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| ① 조명면 찾기 | Sionna 의 Mitsuba/OptiX 광선엔진을 그대로 부른다 — 첫 충돌 탐색과 자기가림 판정이 그쪽 몫이다 |
| ② 면적분 | 그 면 위에서 부품별 재질 PO 를 적분한다 (`src/rcs_sbr.py` `rcs_sbr()`) — E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² |
| 셸 투과 | 얇은 유전체 셸 뒤의 금속(배터리·PCB)을 코히런트 합산한다 (동 `penetrate=True`) |
| 가림의 크기 | 같은 자세를 **다른 팔**로 다시 적분한다 — P(순수 PO 점구름 λ/7, 가림 없음)와 B(SBR+PO 광선격자 λ/12)를 불투명으로 돌린 값의 방위평균 σ 차이를 기체마다 잰다. 셸까지 통과시킨 생산 B 와 P 의 차는 표의 «P − B(생산)» 열이다. 이산화 바닥은 P 팔 한쪽을 λ/7↔λ/12 로 돌린 폭이고 그 옆에 나란히 싣는다 |
| 격자를 무엇에 매나 | 격자 중심·반경·칸수를 자세마다 다시 잡는 팔과 한 판으로 얼린 팔을 같은 씬·같은 자세열에 나란히 태운다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/prior_settled_sionna.json`, `outputs/report3_rt.json`, `outputs/sbr_grid_convergence.json`, `outputs/outofband_power.json`, `outputs/verify_frozen_grid.json`, `outputs/md_classify_verify.json` |
| 소요 | 약 2분 (GPU 0장 — 원장 조립이다) |
| 비고 | σ 격자 자체의 재생성은 `benchmark/rcs_anchor.py` 가 맡는다 |

---

## 두 낱말을 먼저 푼다

**PO** 는 물리광학(physical optics)이다 — 빛이 닿는 면에 흐르는 전류를 근사식으로 바로 적어 넣고 그 면을 훑어 더해 산란을 내는 방법이다. **SBR** 은 광선을 쏴서 튀기며 그 면이 어디인지 찾는 방법(shooting-and-bouncing rays)이다.

상용 고주파 RCS 솔버(FEKO/CST SBR+)의 순서 그대로다 — **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py` `rcs_sbr()`). 레이다식이 표적 산란과 전파 경로를 두 양으로 쓰는 그대로, **σ 는 이 커널이 내고 경로와 환경은 그 엔진이 낸다**.

## 누가 무엇을 하나

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 조명되는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ(θ)\| | 수직입사 보정값 × 각도 모양(TE·TM 전력평균, `ANGLE_GAMMA=1` 기본). 1회 반사 경로 넷이 같은 함수를 쓴다 — 다중반사 경로가 서 있는 자리는 **절 6** 이다 | 🟢 Sionna 재질표(`src/materials.py` `MATERIALS`) + 🔵 각도 모양 (`src/rcs_sbr.py` `ANGLE_GAMMA`) |
| PO 면적분 → σ | E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py` `rcs_sbr()`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB)의 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |

## 왜 우리가 얹어야 하나

Sionna 는 광선을 쏘고 튀긴다 — 기술보고서(v1.2, 59쪽)에 SBR 이 48 회 [^18] 나오고 우리도 그 엔진을 그대로 부른다. 같은 문서에서 `physical optics` 0 회 [^1] · `radar cross section` 0 회 [^19] · `surface current` 0 회 [^20] 이고, 거친 면은 정규화 산란패턴을 쓰는 경험 모델이다 — 그 셈이 어디서 끝나는지는 [부 1 «스톡 엔진이 하는 일과 안 하는 일»](../README.md#부-1-스톡-엔진이-하는-일과-안-하는-일) 가 인자 목록까지 해부했다.

ITU `metal` 의 산란계수 S = 0.0 [^21] 이라 스톡 산란 모델이 금속에서 내놓는 항은 0 이고, 우리 σ 는 면적분에서 창발한다. 금속 4그룹(모터·배터리·PCB·카메라)만 남긴 메쉬의 방위평균 σ 는 전체의 112% [^13] 다.

## PO 적분이 실제로 올라타는 면은 어디까지인가

![mesh_compare_material_shadow](../outputs/figures/mesh_compare_material_shadow.png)

**그림 1.** PO 적분이 실제로 올라타는 면은 어디까지인가?

조명원을 방위 280° [^2] · 고각 15° [^3] 에 두었다 — 방위 72 점 [^22] 스윕에서 7기체 평균 그늘비율의 중앙값에 가장 가까운 방위이고, 규칙이 고른다. 가림 판정은 생산 SBR 이 쓰는 그림자광선 그대로다(`rcs_sbr._exit_visible()`). 그림의 색은 재질이 아니라 조명 상태다.

**표.** 열 이름의 두 팔 — P = 순수 PO(점구름 λ/7), B = SBR+PO(광선격자 λ/12) 이고, B 는 불투명(셸 투과 없음)과 생산(셸 투과) 두 설정으로 선다.

> 원장 정의 그대로다 — 가림 [dB] = 가림 없는 PO 방위평균 σ − 불투명 SBR σ. 셸 투과 [dB] 는 거기서 유전체 셸을 통과시켜 내부 금속을 코히런트 합산했을 때의 차이. 합 = 생산 커널과 순수 PO 의 차이. 이산화 바닥은 PO 를 λ/7↔λ/12 로 돌렸을 때의 폭이다. [^23]

| 기체 | 외피 그늘 | 가림 [dB] | 셸 투과 [dB] | P − B(생산) [dB] | 이산화 바닥 [dB] | 생산 σ [dBsm] |
|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 41 % | +5.98 | +3.66 | +2.32 | 0.014 | -22.0 |
| Mavic 4 Pro | 29 % | +3.75 | +2.55 | +1.20 | 0.021 | -18.2 |
| Matrice 4E ⭐ | 35 % | +6.63 | +3.72 | +2.90 | 0.021 | -18.9 |
| Phantom 4 | 36 % | +5.90 | +4.06 | +1.84 | 0.056 | -19.9 |
| X500 V2 | 47 % | +1.07 | +0.00 | +1.07 | 0.037 | -16.8 |
| Typhoon H (H480) | 39 % | +2.91 | +1.51 | +1.40 | 0.014 | -15.8 |
| S1000+ | 42 % | +0.11 | -0.19 | +0.30 | 0.071 | -12.3 |

출처 [^24]

## 두 팔의 차가 얼마인가

P(순수 PO · 점구름 λ/7 · 가림 없음)와 B(SBR+PO · 광선격자 λ/12)를 불투명으로 돌린 값의 방위평균 σ 차이가 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 가고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 차가 이산화 바닥(최대 0.071 dB [^11]) 위에 있다 — 그 바닥은 **P 팔 한쪽**을 λ/7↔λ/12 로 돌린 폭이고, 원장은 이것을 «가림 효과» 라고 부르기 위한 필요조건으로 쓴다 [^12]. B 팔 광선격자의 자기 불확도는 같은 통계(방위 72 점 [^22] 평균)로 재는 것이 다음 단계 표에 있다.

⚠ 이 표와 그림은 2026-08-04 [^25] 형상 정정 **전** 메쉬 기준이고, 2026-08-07 10:58:22 [^26] Γ(θ) 각도 모양(기본 켬) **이전** 커널의 산출이다 — 가림 최대치를 내는 Matrice 4E 와 X500 V2 가 그 정정을 받은 기체이고, 닫힌 동체의 가림은 셸 형상에 직접 걸린다. 생산 σ 열도 두 축 같은 이유로 재계산 대상이다.

## 격자를 자세마다 다시 정의하면 무엇이 생기나

광선 격자는 표적 앞에 세우는 평면 자다 — 중심 ctr, 반경 Rout, 한 변의 칸수 n 셋이 그것을 정한다. 생산 경로는 그 셋을 **자세마다 bbox 에서 다시 잡는다**. 관절이 도는 로터에서는 bbox 가 자세마다 숨쉬므로 자도 같이 흔들린다.

| 흔들리는 것 | 무엇이 흔들리나 (λ/12 · matrice4e · 4096 자세) | 무엇이 실리나 |
|---|---|---|
| 위상 원점 | ctr 이 시선방향으로 39.9 mm [^27] p-p 돌아다닌다 = 5.85 rad [^28] p-p | 진폭은 4e-16 [^29] 안에서 불변인 채 **위상만** 흔들린다 — 정지한 동체를 시선방향으로 숨쉬게 만드는 것과 같다 |
| 표본 격자 | n = ceil(2Rout/d) 가 정수라 100 [^30]~131 [^31] 사이를 오가고 4095 스텝 중 1636 번 [^32] (40 %) 튄다 | 서브셀 오프셋 표준편차 0.2912 [^33] 는 균등분포 1/√12 = 0.2887 과 넷째 자리까지 같다 — 자세마다 굴리는 **백색 주사위**다 |
| 히트 집합 | 조명된 광선이 평균 610.5 개 [^34], 자세간 상대 표준편차 0.0378 [^35] | 자세별 히트 수 계열 [^36] 의 최대−최소가 102 개(평균의 17 %) 다 — 어느 면이 세어지는가가 자세마다 갈린다 |

## 결정적 검사 — 가산성

서로 가리지 않는 로터의 PO 면적분은 E(φ₁..φ₄) = E₀ + Σ ΔE_j(φ_j) 로 정확히 쪼개진다. 로터 넷을 따로 돌린 합과 넷을 함께 돌린 장의 차이를 잔차로 쓴다 — 이 잣대에는 창도 평활도 분모도 안 들어간다.

| 격자 | 가산성 잔차 (중앙값) | 읽는 법 |
|---|---|---|
| 얼린 판 한 장 | 8.3e-16 [^14] ~ 1.2e-15 [^37] | 기계정밀도 — 정리가 그대로 성립한다 |
| 자세마다 다시 정의 | 0.089 [^38] ~ 1.78 [^15] | O(1) — 물리적으로 결합할 수 없는 로터 사이에 결합이 생긴다 |

그 가짜 결합이 변조로 실린다 — 기체별로 +3.7 [^39] ~ +23.3 dB [^40] 다. 교차 증거로, 광선을 안 쓰는 독립 엔진(순수 PO)과의 대역 안 스펙트럼 일치가 0.440 [^41] 에서 0.953 [^42] 으로 오른다.

## 얼리면 무엇이 오고 무엇을 잃나

잣대를 먼저 정의한다 — **대역밖 전력** P_out 은 슬로타임 스펙트럼에서 블레이드 끝 도플러 f_tip = 1229 Hz [^16] 보다 높은 주파수의 |X(f)|² 를 그대로 더한 값이다 [^43] — 평활도 비율도 안 들어가고, 정규화가 필요하면 분모를 이름에 박아 전체 전력으로 나눈다 [^44].

판 하나를 잡아 4096 자세에 그대로 쓰면 그 대역밖 절대 전력이 λ/12 에서 13.1 [^17] · λ/32 에서 20.1 dB [^45] 내려간다. ⭐ 격자 사다리의 세 팔(생산 · 위상고정 · 얼림 [^46])을 예측 기울기 ≈ −2 [^47] 에 맞대면, div ≥ 12 에서 잰 기울기가 생산 -0.56 [^48] (R² 0.946 [^49]) · 위상고정 -2.33 [^50] (R² 0.848 [^51]) · 얼림 -2.19 [^52] (R² 0.998 [^53]) 다 — 예측 위에 서는 것은 위상고정과 얼림 둘이고, 얼린 팔이 적합도와 절대 바닥(λ/12 에서 P_out 1.3e-08 [^54] 대 위상고정 1.2e-07 [^55])에서 앞선다. 생산 격자는 λ/12 → λ/32 로 촘촘히 해도 2.3 dB [^56] 만 내려간다 — 바닥의 지배 원인이 광선 밀도가 아니라는 뜻이다.

| 대가 | 크기 | 무엇을 뜻하나 |
|---|---|---|
| 광선 수 | 얼린 판이 자세 평균 대비 1.108 배 [^57] | 전 자세를 덮는 판이라 평균보다 크다 — 비용 +10.8 % |
| 디더 평균 | 얼린 장과 생산 장의 레벨 차가 3.35 dB [^58] p-p | 자세별 무작위 오프셋은 사실상 몬테카를로 평균이다. 얼리면 오프셋 한 판에 절대 레벨이 걸린다 — 절대 σ 는 정적 경로에서 가져오고 얼린 복소장은 **모양**에만 쓴다 |
| ⭐그 편향이 삼키는 것 | 판을 반 칸 옮기면 절대 레벨이 3.45 dB [^59] p-p, **두 팔의 차**(가림 축)가 4.16 dB [^60] p-p | 두 팔이 같은 판을 써도 기하가 달라 편향이 공통모드로 빠지지 않는다 — 차가 원본보다 더 흔들린다. 가림 dB 의 **크기**는 [편 38 «동체가 날개를 가리면 변조 깊이와 레벨이 함께…»](38_md-occlusion.ipynb) 에서 판 앙상블 평균이 설 때까지 보류다 |
| 대역 안도 같이 내려간다 | 블레이드 대역 전력 중앙값 -13.9 dB [^61] | 대역밖만 내려가는 것이 아니다 — 아래 절이 그 몫이 잡음이었는지를 광선을 안 쓰는 엔진으로 판정한다 |
| 판을 미리 잡는 일 | 덮개 여유 최소 120.5 mm [^62] | 자세열을 먼저 훑어야 판이 나온다 — 스트리밍으로는 못 잡는다 |

커널의 기본값은 `grid_ref=None` 이다 — 그 값이면 배선 전 커널과 36 [^63]/36 [^64] 비트 동일이라 (최대 상대오차 0.0 [^65]) 옛 원장이 그대로 선다. 판을 잡는 쪽은 호출자다 — 슬로타임 경로(`src/microdoppler.py`)는 로터 한 바퀴의 합집합 경계상자로 판 한 장을 만들어 넘기고, 스위치 `SIONNA2_FREEZE_GRID` 가 그 켬·끔을 정한다(기본 켬). 판이 자세를 못 덮으면 커널이 예외를 던진다 [^66].

⭐ 리포트 6 계열의 마이크로도플러 원장은 얼린 판으로 **다시 났다**(27 열 · 옛 열은 `outputs/prefreeze/` 에 사본으로 남아 있다). 전후 비교와 그 판의 구체적인 수치는 [리포트 6-2 «광선 격자를 어디에 매나»](06_2_engines.ipynb) 가 낸다. ⚠ 아직 옛 판인 것은 바이스태틱 스윕 원장(`report07b_bistatic_md`)과 PO 대조 원장(`report15_po_control`)이다. 여기 숫자는 격자 사다리와 회귀 게이트 자신의 것이다 — 리포트 6 계열의 재계산 결과를 인용하지 않는다.

## 얼리기가 «신호» 도 깎았나 — 광선을 안 쓰는 엔진에게 묻는다

대역밖 전력이 내려간 것은 좋은 소식이다. 그런데 같은 재계산에서 **블레이드 대역 안**의 전력도 함께 내려갔다 — 중앙값 -13.9 dB [^61] 다. 그 몫이 표적의 진짜 운동이었다면 얼리기는 물리를 지운 것이다. 그래서 판정을 **광선 격자를 안 쓰는 엔진**에게 맡긴다 — 순수 PO 는 점구름 면적분이라 격자가 없고, 이번 재계산에서 비트 그대로였다 [^67].

| 잣대 | 얼리기 전 | 얼린 뒤 | 무엇을 뜻하나 |
|---|---|---|---|
| 블레이드 대역이 순수 PO 보다 몇 dB 위인가 | +20.3 dB [^68] | +4.0 dB [^69] | PO **밑으로** 내려간 열이 0 [^70] 개다 — 깎인 것은 잉여였다 |
| 같은 대역 복소 파형이 PO 와 닮은 정도 | 0.34 [^71] | 0.75 [^72] | 12 [^73]/13 [^74] 열에서 올랐다 — 전력이 줄면서 닮음이 오르면 줄어든 것은 잡음이다 |
| 플래시 대조비 (봉우리 ÷ 바닥) | — | +3.0 dB [^75] (중앙값) | 18 [^76]/27 [^77] 열에서 올랐다 — 자의 흔들림이 골짜기를 메우고 있었다는 뜻이다 |
| 플래시 주파수 추정의 상대오차 | 0.31% [^78] | 0.31% [^79] | 포락 자기상관으로 읽은 f_flash — 두 판 다 예측 위에 선다 |
| 스펙트럼 가장자리 ÷ f_tip (운동학이 맞으면 1) | 4.33 [^80] | 2.96 [^81] | 순수 PO 는 0.93 [^82] 다 — ⚠14 [^83] 열은 얼린 뒤 1 에서 더 멀어진다. 폭 지표는 PO 열에서 읽는다 |

⚠ **대가는 비율 쪽에 있다.** 절대 대역밖 전력은 내려가지만 블레이드 대역이 더 많이 내려가는 열이 있어서, «신호 대 비물리 잔차»(P_out/P_in)로 읽으면 동체가 든 15 [^84] 열 중 13 [^85] 열이 -8.8 dB [^86] → -6.2 dB [^87] 로 **나빠진다**(순수 PO 는 -29.6 dB [^88]). 잉여가 사라진 만큼 남은 잔차가 상대적으로 커 보이는 것이고, 그 잔차는 여전히 물리가 아니다 — 지금 판이 마이크로도플러에 줄 수 있는 동적범위의 상한이 여기 있다.

반대로 블레이드만 남긴 12 [^89] 열은 빗살 대조비가 12 [^90]/12 [^89] 열에서 +7.8 dB [^91] 올라온다 — 신호가 약할수록 얼리기가 크게 남는다 [^92].

⚠ 심판이 아직 붙지 않은 자리가 하나 남는다 [^93] — 2 초 호버 두 열은 순수 PO 대조 팔을 기다린다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 얼린 판으로 **바이스태틱** 스윕 원장과 PO 대조 원장을 다시 낸다 | 모노 쪽은 갈아탔고(리포트 6-2), 남은 두 원장만 아직 옛 판이라 두 편의 절대값을 이어 붙여 읽을 수 없다 | `src/microdoppler.py`(판을 넘기는 쪽) → [편 35 «시간표본마다 자세를 새로 놓고 다시 쏘아 슬로…»](35_md-slowtime.ipynb) |
| B 팔 광선격자의 자기 불확도를 P 팔 바닥과 **같은 통계**(방위 72점 평균 σ)로 잰다 | 가림 축의 판정 바닥이 두 팔 모두에서 서고, 어느 기체가 판정 안에 드는지가 확정된다 | `src/viz_mesh_material.py` 가림 축 → outputs/report02_derived.json `occlusion.floor_db` |
| 정정된 메쉬로 가림 표와 생산 σ 를 같은 설정에서 다시 낸다 | 형상 정정이 가림과 σ 를 어느 방향으로 얼마나 옮기는지가 기체별로 확정된다 | [^94] |
| 같은 메쉬를 스톡 경로 솔버에 그대로 넣고 무엇이 나오는지 잰다 | 우리 커널이 스톡 위에 얹은 항이 무엇인지가 나란히 확정된다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |
| 수신 방향 그림자 광선을 켜고 바이스태틱으로 넓힌다 | 출사 쪽 가림이 상반성 위반을 얼마나 줄이는지가 확정된다 | [편 20 «수신 방향 그림자 광선을 켜면 상반성 위반이…»](20_bistatic-exit.ipynb) |
| PO 면적분을 디바이스 커널로 옮긴다 | 전격자 재생성 비용이 확정된다 — 지금은 호스트가 대부분을 쓴다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 94개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics` | 0 |
| [^2] | `outputs/report02_derived.json` | `occlusion.az_deg` | 280 |
| [^3] | `outputs/report02_derived.json` | `occlusion.el_deg` | 15 |
| [^4] | `outputs/report02_derived.json` | `occlusion.shadow_min_pct` | 29.06 |
| [^5] | `outputs/report02_derived.json` | `occlusion.shadow_max_pct` | 46.54 |
| [^6] | `outputs/report02_derived.json` | `occlusion.max_db` | 6.626 |
| [^7] | `outputs/report02_derived.json` | `occlusion.max_drone` | Matrice 4E ⭐ |
| [^8] | `outputs/report02_derived.json` | `occlusion.min_drone` | S1000+ |
| [^9] | `outputs/report02_derived.json` | `occlusion.min_db` | 0.1111 |
| [^10] | `outputs/report02_derived.json` | `occlusion.n_above_floor` | 7 |
| [^11] | `outputs/report02_derived.json` | `occlusion.floor_max_db` | 0.07061 |
| [^12] | `outputs/mesh_compare_material.json` | `airframes.mini5pro.sigma.caveat` | PO 와 SBR 은 이산화가 다르다(점구름 λ/7 vs 광선격자 λ/12·jitter 2). dis… |
| [^13] | `outputs/report3_rt.json` | `C_metal.metal_share_pct` | 112 |
| [^14] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.pinned.additivity_residual_median` | 8.282e-16 |
| [^15] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.moving.additivity_residual_median` | 1.781 |
| [^16] | `outputs/outofband_power.json` | `_meta.f_tip_hz` | 1229 |
| [^17] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.12` | 13.09 |
| [^18] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing` | 48 |
| [^19] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section` | 0 |
| [^20] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.surface current` | 0 |
| [^21] | `outputs/report3_rt.json` | `C_metal.itu_metal_S` | 0 |
| [^22] | `outputs/report02_derived.json` | `occlusion.n_az_sweep` | 72 |
| [^23] | `outputs/report02_derived.json` | `occlusion.definition` | 가림 [dB] = 가림 없는 PO 방위평균 σ − 불투명 SBR σ. 셸 투과 [dB] 는 거기서… |
| [^24] | `outputs/report02_derived.json` | `occlusion.rows` | (7행 표) |
| [^25] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^26] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^27] | `outputs/sbr_grid_convergence.json` | `grid_wander.ctr_u_ptp_mm` | 39.89 |
| [^28] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.ctr_dot_u_ptp_rad` | 5.852 |
| [^29] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.rows[1].max_abs_change` | 3.906e-16 |
| [^30] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_min` | 100 |
| [^31] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_max` | 131 |
| [^32] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_changes` | 1636 |
| [^33] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.subcell_off_e1_std_frac` | 0.2912 |
| [^34] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_mean` | 610.5 |
| [^35] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_relstd` | 0.03777 |
| [^36] | `outputs/sbr_grid_convergence.npz` | `n_lit_div12` | (4096행 표) |
| [^37] | `outputs/md_classify_verify.json` | `grid_pinning.mini5pro.pinned.additivity_residual_median` | 1.167e-15 |
| [^38] | `outputs/md_classify_verify.json` | `grid_pinning.s1000plus.moving.additivity_residual_median` | 0.08905 |
| [^39] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.spurious_modulation_db` | 3.651 |
| [^40] | `outputs/md_classify_verify.json` | `grid_pinning.phantom4.spurious_modulation_db` | 23.31 |
| [^41] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_prod_vs_po` | 0.4402 |
| [^42] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_froz_vs_po` | 0.9535 |
| [^43] | `outputs/outofband_power.json` | `new_definition.headline` | P_out = Σ \|X(f)\|² over \|f\| > f_tip   (평활 없음, 원 주기도) |
| [^44] | `outputs/outofband_power.json` | `new_definition.normalization` | 필요할 때만, **전체 전력**으로: frac_of_total = P_out / P_tot. 분모를… |
| [^45] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.32` | 20.13 |
| [^46] | `outputs/sbr_grid_convergence.json` | `_meta.arms` | (3항목 묶음) |
| [^47] | `outputs/sbr_grid_convergence.json` | `_meta.prediction` | 실루엣이 격자를 가로지르며 광선이 토글 → 잡음 ∝ d² → 대역밖 비율의 log-log 기울기 ≈… |
| [^48] | `outputs/outofband_power.json` | `convergence.prod.slope_ge12` | -0.5604 |
| [^49] | `outputs/outofband_power.json` | `convergence.prod.r2_ge12` | 0.9463 |
| [^50] | `outputs/outofband_power.json` | `convergence.phase.slope_ge12` | -2.33 |
| [^51] | `outputs/outofband_power.json` | `convergence.phase.r2_ge12` | 0.848 |
| [^52] | `outputs/outofband_power.json` | `convergence.froz.slope_ge12` | -2.191 |
| [^53] | `outputs/outofband_power.json` | `convergence.froz.r2_ge12` | 0.9981 |
| [^54] | `outputs/outofband_power.json` | `convergence.froz.P_out_per_div.12` | 1.259e-08 |
| [^55] | `outputs/outofband_power.json` | `convergence.phase.P_out_per_div.12` | 1.152e-07 |
| [^56] | `outputs/outofband_power.json` | `convergence.prod.drop_db_div12_to_div32` | 2.297 |
| [^57] | `outputs/verify_frozen_grid.json` | `gate2_frozen_grid_invariant.extra_ray_cost` | 1.108 |
| [^58] | `outputs/verify_frozen_grid.json` | `field_level.froz_vs_prod_level_db_ptp` | 3.351 |
| [^59] | `outputs/freeze_plate_sensitivity.json` | `verdict.abs_level_plate_ptp_db` | 3.452 |
| [^60] | `outputs/freeze_plate_sensitivity.json` | `verdict.occlusion_level_plate_ptp_db` | 4.163 |
| [^61] | `outputs/freeze_signal_loss.json` | `summary.P_in_delta_db_median` | -13.87 |
| [^62] | `outputs/verify_frozen_grid.json` | `gate3_coverage.margin_min_mm` | 120.5 |
| [^63] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_bit_identical` | 36 |
| [^64] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_cases` | 36 |
| [^65] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.max_rel_err` | 0 |
| [^66] | `outputs/verify_frozen_grid.json` | `negative_controls.too_small_msg` | sbr_field: 얼린 격자가 이 자세를 못 덮는다 — 여유 e1 -161.21 · e2 +43.… |
| [^67] | `outputs/freeze_signal_loss.json` | `_meta.independent_judge_ko` | 순수 PO 열(`C_po_locked`·`D_po_spread`·`report07_three_eng… |
| [^68] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.before_median` | 20.3 |
| [^69] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.after_median` | 4.019 |
| [^70] | `outputs/freeze_signal_loss.json` | `summary.P_in_excess_over_po_db.n_below_po_after` | 0 |
| [^71] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.before_median` | 0.3429 |
| [^72] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.after_median` | 0.7489 |
| [^73] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.n_improved` | 12 |
| [^74] | `outputs/freeze_signal_loss.json` | `summary.blade_coh_vs_po.n_judged` | 13 |
| [^75] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.delta_median` | 2.979 |
| [^76] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.n_improved` | 18 |
| [^77] | `outputs/freeze_signal_loss.json` | `summary.flash_contrast_db.n_series` | 27 |
| [^78] | `outputs/freeze_signal_loss.json` | `summary.f_flash_relerr_abs_median.before` | 0.00313 |
| [^79] | `outputs/freeze_signal_loss.json` | `summary.f_flash_relerr_abs_median.after` | 0.00313 |
| [^80] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.before_median` | 4.329 |
| [^81] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.after_median` | 2.965 |
| [^82] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.po_median` | 0.93 |
| [^83] | `outputs/freeze_signal_loss.json` | `summary.width_ratio_20db.n_farther_from_one` | 14 |
| [^84] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.n` | 15 |
| [^85] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.n_oob_worse` | 13 |
| [^86] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.oob_over_in_db_before_median` | -8.814 |
| [^87] | `outputs/freeze_signal_loss.json` | `summary.by_group.with_body.oob_over_in_db_after_median` | -6.245 |
| [^88] | `outputs/freeze_signal_loss.json` | `summary.oob_over_in_db.po_median` | -29.63 |
| [^89] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.n` | 12 |
| [^90] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.n_comb_improved` | 12 |
| [^91] | `outputs/freeze_signal_loss.json` | `summary.by_group.blade_only.comb_line_delta_db_median` | 7.798 |
| [^92] | `outputs/freeze_signal_loss.json` | `summary.by_group.why_ko` | 동체가 든 열(A·B·2 초 호버·세 엔진)은 동체 DC 가 크고 가짜 변조도 거기에 실려 있었다… |
| [^93] | `outputs/freeze_signal_loss.json` | `verdict.open_ko` | ⚠ `report07_hover_long*` 두 열에는 **순수 PO 대조 팔이 없다**. 2 초… |
| [^94] | `outputs/meshfix_attack.json` | `recommended_gate_before_any_sigma_claim` | (6행 표) |